# Multistage Pipeline — HiLift Setup

Platform-team setup notebook for the **multistage training pipeline**
demo. The pipeline itself (in
`advanced_multistage_training_pipeline.ipynb`) is a chain of three
independent training "teams" — **init → pretrain → posttrain** —
that exchange state exclusively through the W&B Registry. This
notebook publishes the fixed inputs both ends of that pipeline
need:

1. **One untrained `Transolver++` base model** —
   `hilift_multistage_models/base:production`. Stage 1 loads it as
   the starting `state_dict`.
2. **Three purpose-differentiated datasets** under
   `hilift_multistage_datasets`:
   - `synthetic:production` — LHC001 × [4°, 8°, 12°, 16°] = 4 cases
     (single-geometry warm-up).
   - `pretraining:production` — all 3 geometries × [4°, 8°, 12°]
     = 9 cases (broad geometry, low/mid AoA).
   - `posttraining:production` — all 3 geometries × [16°] = 3 cases
     (high-AoA / near-stall focus).

The three datasets follow Hodges' framing: warm-up on one geometry,
broaden across geometries, then focus on the aerodynamically
interesting regime.

**Own Registries.** This demo uses **two dedicated Registries** —
`hilift_multistage_datasets` and `hilift_multistage_models` — kept
separate from the workshop's main `hilift` Registry. Different
asset types → different Registries is a pedagogical point (ACLs,
governance, alias namespaces).

**Prerequisites**
- Access to the `wandb_emea` entity (or edit `ENTITY` in §1).
- Internet — §2 clones `thuml/Transolver_plus` from GitHub if the
  workshop's main training notebook hasn't already cached it.

## 1. Setup

Third-party imports and the section-local constants.


In [1]:
from __future__ import annotations

import importlib
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np

import torch
import torch.nn as nn

import wandb

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")


In [2]:
# ---- W&B target ----
ENTITY = "wandb_emea"
PROJECT = "jb_physics_demo"

# ---- Two dedicated Registries ----
DATASETS_REG = "hilift_multistage_datasets"
MODELS_REG   = "hilift_multistage_models"

# ---- Per-Collection dataset specs ----
# Each list follows the (geo_id, aoa) tuple format used elsewhere
# in the workshop. Curriculum:
#   synthetic:    single geometry, all AoAs — warm-up.
#   pretraining:  all geometries, low/mid AoA — broad domain.
#   posttraining: all geometries, high AoA only — target regime.
CASE_SPECS_BY_COLLECTION: dict[str, list[tuple[str, int]]] = {
    "synthetic": [
        ("LHC001",  4), ("LHC001",  8), ("LHC001", 12), ("LHC001", 16),
    ],
    "pretraining": [
        ("LHC001",  4), ("LHC001",  8), ("LHC001", 12),
        ("LHC002",  4), ("LHC002",  8), ("LHC002", 12),
        ("LHC003",  4), ("LHC003",  8), ("LHC003", 12),
    ],
    "posttraining": [
        ("LHC001", 16), ("LHC002", 16), ("LHC003", 16),
    ],
}

POINTS_PER_CASE_CACHE = 8192

# ---- Base model architecture (matches experiment_tracking_hilift.ipynb defaults) ----
BASE_MODEL_ARCH = {
    "feature_dim": 6,
    "out_dim":     1,
    "n_hidden":   96,
    "n_layers":    4,
    "n_head":      4,
    "slice_num":  16,
    "mlp_ratio":   2,
    "dropout":  0.05,
}

TMP_DIR = Path(tempfile.mkdtemp(prefix="hilift_multistage_setup_"))

# ---- Collection card content (rendered on the Registry Collection page) ----
BASE_MODEL_DESCRIPTION = """\
# HiLift Transolver++ — untrained base model

Reference architecture for the multistage pipeline. Consumed by
**Stage 1** of `advanced_multistage_training_pipeline.ipynb` as the
starting `state_dict`.

**Schema:** checkpoint dict with `state_dict`, `arch` (Transolver++
kwargs), `stage="base"`, `trained=False`.

**Update cadence:** stable — only re-published if the reference
architecture changes.
"""

COLLECTION_DESCRIPTIONS: dict[str, str] = {
    "synthetic": """\
# HiLift multistage — synthetic warm-up dataset

**4 cases** — LHC001 × [4°, 8°, 12°, 16°]. Single-geometry
introduction to the CRM-HL topology. **Stage 1** derives the
canonical normalisation stats from this Collection, so every
downstream stage inherits stats grounded in `synthetic`.

**Schema:** one NPZ per case (`points`, `normals`, `target`,
`geo_id`, `aoa`, `run_id`, ...).

**Consumer:** `advanced_multistage_training_pipeline.ipynb` §3.
""",
    "pretraining": """\
# HiLift multistage — pretraining dataset

**9 cases** — all 3 geometries × [4°, 8°, 12°]. Broad-domain
coverage for **Stage 2** to build geometry generalisation on top of
Stage 1's warm-started weights.

**Schema:** same NPZ shape as `synthetic`.

**Consumer:** `advanced_multistage_training_pipeline.ipynb` §4.
""",
    "posttraining": """\
# HiLift multistage — posttraining dataset

**3 cases** — all 3 geometries × [16°]. The high-AoA / near-stall
regime; the aerodynamically interesting slice where **Stage 3**
fine-tunes the pretrained model.

**Schema:** same NPZ shape as `synthetic`.

**Consumer:** `advanced_multistage_training_pipeline.ipynb` §5.
""",
}


def write_collection_card(target_path: str, description: str, coll_type: str) -> None:
    """Set the markdown card on a Registry Collection. Idempotent.

    Runs after `link_artifact` (which implicitly creates the
    Collection). Wrapped in try/except so wandb-version differences
    in the API don't block publishing.
    """
    try:
        coll = wandb.Api().artifact_collection(
            type_name=coll_type, name=target_path,
        )
        coll.description = description
        coll.save()
        print(f"    wrote Collection card ({len(description)} chars)")
    except Exception as exc:
        print(f"    Collection card skipped: {exc}")


print(f"wandb          = {wandb.__version__}")
print(f"target project = {ENTITY}/{PROJECT}")
print(f"datasets reg   = wandb-registry-{DATASETS_REG}/(synthetic,pretraining,posttraining)")
print(f"models reg     = wandb-registry-{MODELS_REG}/base")
print(f"tmp            = {TMP_DIR}")


wandb          = 0.28.0
target project = wandb_emea/jb_physics_demo
datasets reg   = wandb-registry-hilift_multistage_datasets/(synthetic,pretraining,posttraining)
models reg     = wandb-registry-hilift_multistage_models/base
tmp            = /var/folders/dm/kyhk1vhx7jb_k1b2z7zdns6c0000gp/T/hilift_multistage_setup_f21_skdo


## 2. Demo generator + base model architecture

The workshop's demo-synthetic point-cloud generator (from
`registry.ipynb`) is reused verbatim — the multistage demo doesn't
introduce a new data-shape, just a curriculum over which
`(geo_id, aoa)` cases each stage sees. Base-model architecture
matches the workshop's default `TransolverPointFieldSurrogate`
config so downstream stages can `load_state_dict` into a
freshly-built model without shape drift.

Dataset spec at a glance:

| Collection      | # cases | geometries              | AoAs (°)              |
| --------------- | ------- | ----------------------- | --------------------- |
| `synthetic`     |    4    | LHC001                  | 4, 8, 12, 16          |
| `pretraining`   |    9    | LHC001, LHC002, LHC003  | 4, 8, 12              |
| `posttraining`  |    3    | LHC001, LHC002, LHC003  | 16                    |


In [3]:
# ==== §2 — Transolver++ clone/patch + generator + base model ==========

# ---- Transolver++ clone + patch (reused cache if already present) ----
CACHE_ROOT = Path.home() / ".cache" / "physicsx_demo"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
TRANSOLVER_DIR = CACHE_ROOT / "Transolver_plus"

if not TRANSOLVER_DIR.exists():
    print(f"Cloning thuml/Transolver_plus into {TRANSOLVER_DIR} ...")
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/thuml/Transolver_plus.git",
        str(TRANSOLVER_DIR),
    ])
else:
    print(f"Using cached Transolver_plus at {TRANSOLVER_DIR}")

model_file = TRANSOLVER_DIR / "models" / "Transolver_plus.py"
src_text = model_file.read_text()
if "def _maybe_all_reduce" not in src_text:
    src_text = src_text.replace(
        "import torch.distributed.nn as dist_nn\n",
        "import torch.distributed.nn as dist_nn\n"
        "import torch.distributed as dist\n\n\n"
        "def _maybe_all_reduce(tensor):\n"
        "    if dist.is_available() and dist.is_initialized() "
        "and dist.get_world_size() > 1:\n"
        "        dist_nn.all_reduce(tensor, op=dist_nn.ReduceOp.SUM)\n"
        "    return tensor\n",
    )
    src_text = src_text.replace(
        "dist_nn.all_reduce(slice_norm, op=dist_nn.ReduceOp.SUM)",
        "_maybe_all_reduce(slice_norm)",
    )
    src_text = src_text.replace(
        "dist_nn.all_reduce(slice_token, op=dist_nn.ReduceOp.SUM)",
        "_maybe_all_reduce(slice_token)",
    )
    model_file.write_text(src_text)
    print("Patched Transolver_plus.py with _maybe_all_reduce.")
else:
    print("Transolver_plus.py already patched.")

if str(TRANSOLVER_DIR) not in sys.path:
    sys.path.insert(0, str(TRANSOLVER_DIR))
if "models.Transolver_plus" in sys.modules:
    importlib.reload(sys.modules["models.Transolver_plus"])
from models.Transolver_plus import Model as TransolverPlusModel


# ---- Model wrapper (matches experiment_tracking_hilift.ipynb) ----
class TransolverPointFieldSurrogate(nn.Module):
    """Point-cloud → per-point scalar prediction via Transolver++."""

    def __init__(
        self,
        feature_dim: int = 6,
        out_dim: int = 1,
        n_hidden: int = 96,
        n_layers: int = 4,
        n_head: int = 4,
        slice_num: int = 16,
        mlp_ratio: int = 2,
        dropout: float = 0.05,
    ) -> None:
        super().__init__()
        self.backbone = TransolverPlusModel(
            space_dim=feature_dim, fun_dim=0, out_dim=out_dim,
            n_hidden=n_hidden, n_layers=n_layers, n_head=n_head,
            mlp_ratio=mlp_ratio, slice_num=slice_num,
            dropout=dropout, unified_pos=False,
        )

    def forward(
        self,
        x: torch.Tensor,
        pos: torch.Tensor,
        condition: torch.Tensor,
    ) -> torch.Tensor:
        return self.backbone((x, pos, condition))


# ---- Demo point-cloud generator (verbatim from registry.ipynb) ----
def _run_id(geo_id: str, aoa: float | int) -> str:
    """Canonical run/case identifier — matches the reference notebook's shape."""
    return f"geo_{geo_id}_AoA_{int(aoa)}"


def create_demo_pointcloud_case(
    geo_id: str,
    aoa: float | int,
    n_points: int,
    seed: int,
) -> dict:
    """Build one synthetic slat/main-wing/flap point-cloud case."""
    rid = f"demo_{_run_id(geo_id, aoa)}"
    rng_seed = seed + int(float(aoa) * 10) + sum(ord(ch) for ch in geo_id)
    rng = np.random.default_rng(rng_seed)
    n = n_points

    component = rng.choice(3, size=n, p=[0.22, 0.56, 0.22])

    y = rng.uniform(-1.0, 1.0, size=n)
    span = np.abs(y)
    chord = 1.0 - 0.32 * span
    sweep = 0.22 * span

    geo_num = int("".join(ch for ch in geo_id if ch.isdigit()) or 1)
    geo_phase = 0.5 + 0.5 * np.sin(geo_num * 1.7)

    x = np.zeros(n, dtype=np.float32)
    z = np.zeros(n, dtype=np.float32)
    normals = np.zeros((n, 3), dtype=np.float32)

    slat = component == 0
    main = component == 1
    flap = component == 2

    u = rng.uniform(0.04, 0.96, size=main.sum())
    x[main] = sweep[main] + u * chord[main]
    z[main] = 0.055 * np.sin(np.pi * u) * (1.0 - 0.25 * span[main])
    normals[main] = np.array([0.0, 0.0, 1.0], dtype=np.float32)

    u = rng.uniform(-0.20, 0.05, size=slat.sum())
    x[slat] = sweep[slat] + u * chord[slat]
    z[slat] = 0.075 + 0.025 * np.cos((u + 0.20) / 0.25 * np.pi) * (1.0 - 0.2 * span[slat])
    normals[slat] = np.array([-0.20, 0.0, 0.98], dtype=np.float32)

    u = rng.uniform(0.90, 1.25, size=flap.sum())
    flap_deflection = np.deg2rad(22.0 + 0.35 * float(aoa) + 7.0 * geo_phase)
    x[flap] = sweep[flap] + u * chord[flap]
    z[flap] = -0.035 - np.sin(flap_deflection) * (u - 0.90) * 0.55
    z[flap] += 0.025 * np.sin(np.pi * np.clip((u - 0.90) / 0.35, 0, 1)) * (1.0 - 0.25 * span[flap])
    normals[flap] = np.array(
        [np.sin(flap_deflection), 0.0, np.cos(flap_deflection)],
        dtype=np.float32,
    )

    points = np.stack([x, 3.2 * y, z], axis=1).astype(np.float32)
    normals /= np.maximum(np.linalg.norm(normals, axis=1, keepdims=True), 1e-6)

    aoa_scale = float(aoa) / 16.0
    leading_edge = sweep + 0.08 * chord

    le_suction = (
        -(0.65 + 0.75 * aoa_scale)
        * np.exp(-((x - leading_edge) ** 2) / 0.012)
        * (1.0 - 0.28 * span)
    )
    flap_loading = (
        -0.45
        * np.exp(-((x - (sweep + 1.03 * chord)) ** 2) / 0.035)
        * flap.astype(float)
    )
    slat_loading = (
        -0.38
        * np.exp(-((x - (sweep - 0.08 * chord)) ** 2) / 0.018)
        * slat.astype(float)
    )
    pressure_recovery = 0.18 * x + 0.035 * y + 0.06 * geo_phase
    noise = rng.normal(0.0, 0.015, size=n)

    target = (
        le_suction + flap_loading + slat_loading + pressure_recovery + noise
    ).astype(np.float32)[:, None]

    return {
        "points": points,
        "normals": normals.astype(np.float32),
        "target": target,
        "geo_id": geo_id,
        "aoa": float(aoa),
        "run_id": rid,
        "target_name": "synthetic_pressure_like_cp",
        "target_location": "generated_demo",
        "target_mode": "scalar",
    }


def save_dataset_cases(
    specs: list[tuple[str, int]],
    out_dir: Path,
    seed: int,
    points_per_case: int,
) -> list[Path]:
    """Materialise one Collection's NPZs from a (geo_id, aoa) spec list."""
    out_dir.mkdir(parents=True, exist_ok=True)
    paths: list[Path] = []
    for i, (geo_id, aoa) in enumerate(specs):
        case = create_demo_pointcloud_case(
            geo_id=geo_id, aoa=aoa,
            n_points=points_per_case, seed=seed,
        )
        p = out_dir / f"case_{i:03d}_{case['run_id']}.npz"
        np.savez_compressed(
            p,
            points=case["points"],
            normals=case["normals"],
            target=case["target"],
            geo_id=np.array(case["geo_id"]),
            aoa=np.array(case["aoa"], dtype=np.float32),
            run_id=np.array(case["run_id"]),
            target_name=np.array(case["target_name"]),
            target_location=np.array(case["target_location"]),
            target_mode=np.array(case["target_mode"]),
        )
        paths.append(p)
    return paths


# ---- Instantiate untrained base model and serialise to disk ----
base_model = TransolverPointFieldSurrogate(**BASE_MODEL_ARCH)
base_ckpt_path = TMP_DIR / "base.pt"
torch.save(
    {
        "state_dict": base_model.state_dict(),
        "arch": dict(BASE_MODEL_ARCH),
        "stage": "base",
        "trained": False,
        "created_by": "advanced_multistage_setup.ipynb",
    },
    base_ckpt_path,
)
n_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
print(f"base model:   {n_params:,} trainable params")
print(f"base ckpt:    {base_ckpt_path}")


Using cached Transolver_plus at /Users/jbutt/.cache/physicsx_demo/Transolver_plus
Transolver_plus.py already patched.
base model:   255,477 trainable params
base ckpt:    /var/folders/dm/kyhk1vhx7jb_k1b2z7zdns6c0000gp/T/hilift_multistage_setup_f21_skdo/base.pt


/Users/jbutt/Developer/physicsx_demo/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Publish base model + three datasets

One platform-team `wandb.init(job_type="publish-base")` block covers
the whole thing:

1. Idempotently create both Registries (`api.create_registry(...)`
   inside `try/except`).
2. Log + link the untrained base model to
   `wandb-registry-hilift_multistage_models/base` with aliases
   `production`, `v1`.
3. For each of the three dataset specs, materialise its NPZs on
   disk, wrap them into a `wandb.Artifact(type="dataset")`, log +
   `.wait()` + link to
   `wandb-registry-hilift_multistage_datasets/{name}` with aliases
   `production`, `v1`.

Collections are created implicitly on first link — no explicit
`create_collection` call needed. Downstream teams consume by
canonical path (`wandb-registry-hilift_multistage_{...}/{name}:production`).


In [4]:
with wandb.init(
    entity=ENTITY, project=PROJECT,
    job_type="publish-base",
    name="hilift-multistage-publish-base",
    tags=["notebook", "advanced", "multistage", "setup"],
) as run:
    api = wandb.Api()

    # ---- Ensure both Registries exist (idempotent) ----
    for reg_name in (DATASETS_REG, MODELS_REG):
        try:
            api.create_registry(name=reg_name, visibility="organization")
            print(f"created registry: {reg_name}")
        except Exception as exc:
            print(f"registry {reg_name} create skipped: {exc}")

    # ---- Publish base model ----
    base_art = wandb.Artifact(
        name="hilift_transolverpp_base",
        type="model",
        description=(
            "Untrained TransolverPointFieldSurrogate at the workshop's "
            "default architecture. Consumed by Stage 1 of the multistage "
            "pipeline as the starting state_dict."
        ),
        metadata={
            "arch": dict(BASE_MODEL_ARCH),
            "stage": "base",
            "trained": False,
            "n_params": n_params,
        },
    )
    base_art.add_file(str(base_ckpt_path), name="checkpoint.pt")
    logged_base = run.log_artifact(base_art)
    logged_base.wait()

    base_target = f"wandb-registry-{MODELS_REG}/base"
    run.link_artifact(
        logged_base, target_path=base_target,
        aliases=["production", "v1"],
    )
    print()
    print(f"published base model -> {base_target}:production")
    write_collection_card(base_target, BASE_MODEL_DESCRIPTION, coll_type="model")

    # ---- Publish three datasets ----
    for collection_name, specs in CASE_SPECS_BY_COLLECTION.items():
        ds_dir = TMP_DIR / f"dataset_{collection_name}"
        paths = save_dataset_cases(
            specs=specs, out_dir=ds_dir,
            seed=SEED, points_per_case=POINTS_PER_CASE_CACHE,
        )

        ds_art = wandb.Artifact(
            name=f"hilift_multistage_{collection_name}",
            type="dataset",
            description=(
                f"HiLift multistage — '{collection_name}' Collection "
                f"({len(specs)} cases). Demo-synthetic point clouds "
                f"with pressure-like Cp target; same schema as the "
                f"workshop's main hilift dataset."
            ),
            metadata={
                "collection": collection_name,
                "n_cases": len(specs),
                "points_per_case": POINTS_PER_CASE_CACHE,
                "case_specs": [list(s) for s in specs],
                "geometries": sorted({g for g, _ in specs}),
                "aoas": sorted({a for _, a in specs}),
                "target_name": "synthetic_pressure_like_cp",
                "seed": SEED,
            },
        )
        for p in paths:
            ds_art.add_file(str(p), name=p.name)

        logged_ds = run.log_artifact(ds_art)
        logged_ds.wait()

        ds_target = f"wandb-registry-{DATASETS_REG}/{collection_name}"
        run.link_artifact(
            logged_ds, target_path=ds_target,
            aliases=["production", "v1"],
        )
        print(
            f"published {len(specs):>2} cases -> {ds_target}:production   "
            f"(geos={sorted({g for g, _ in specs})}, "
            f"aoas={sorted({a for _, a in specs})})"
        )
        write_collection_card(
            ds_target,
            COLLECTION_DESCRIPTIONS[collection_name],
            coll_type="dataset",
        )

    print()
    print(f"run URL -> {run.url}")


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: jbutt (wandb_emea) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


created registry: hilift_multistage_datasets
created registry: hilift_multistage_models


wandb: WARNING Artifact "hilift_transolverpp_base" already exists with the same content. No new version will be created.



published base model -> wandb-registry-hilift_multistage_models/base:production
    wrote Collection card (399 chars)
published  4 cases -> wandb-registry-hilift_multistage_datasets/synthetic:production   (geos=['LHC001'], aoas=[4, 8, 12, 16])
    wrote Collection card (447 chars)
published  9 cases -> wandb-registry-hilift_multistage_datasets/pretraining:production   (geos=['LHC001', 'LHC002', 'LHC003'], aoas=[4, 8, 12])
    wrote Collection card (313 chars)
published  3 cases -> wandb-registry-hilift_multistage_datasets/posttraining:production   (geos=['LHC001', 'LHC002', 'LHC003'], aoas=[16])
    wrote Collection card (316 chars)

run URL -> https://wandb.ai/wandb_emea/jb_physics_demo/runs/uv7eagqx


## 4. Wrap up — what to look at next

**In the W&B UI**, browse to the org-level Registry page. You
should see two new Registries:

- **`hilift_multistage_datasets`** with three Collections
  (`synthetic`, `pretraining`, `posttraining`), each containing
  one version aliased `production` + `v1`, and each with a
  **Collection card** (rendered markdown at the top of the
  Collection page) describing purpose, schema, and consumer.
- **`hilift_multistage_models`** with one Collection (`base`),
  containing one version aliased `production` + `v1`, also
  carded.

Cards are written via the Public API
(`api.artifact_collection(...).description = "..."; .save()`) at
the end of the publish run. They're idempotent — re-running this
notebook re-syncs the card content with whatever's in the
`COLLECTION_DESCRIPTIONS` dict / `BASE_MODEL_DESCRIPTION` string.

The `hilift-multistage-publish-base` run under
`wandb_emea/jb_physics_demo` should have all four artifacts on its
artifacts tab, one produced-edge each.

**Next step.** Open
`notebooks/advanced_multistage_training_pipeline.ipynb`. That
notebook picks up here: three sequential training stages, each
consuming Registry entries produced by this notebook, promoting
their outputs into new Collections (`initialised`, `pretrained`,
`posttrained`) under `hilift_multistage_models`.
